In [1]:
!pip install openmeteo_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.8/707.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.7 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [2]:
import openmeteo_requests
from datetime import datetime


class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed + self.step > self.max_speed:
            raise StopIteration

        self.current_speed += self.step
        return self.current_speed


class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed=0, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed - self.step < self.min_speed:
            raise StopIteration

        self.current_speed -= self.step
        return self.current_speed


class Car:

    cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):

        self.max_speed = max_speed
        self.current_speed = current_speed

        if current_speed == 0:
            self.state = "parking"
        else:
            self.state = "on_road"
            Car.cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):

        if self.state == "parking":
            self.state = "on_road"
            Car.cars_on_road += 1

        start_speed = self.current_speed

        increaser = IncreaseSpeed(self.current_speed, self.max_speed, step)

        if upper_border is not None:

            for speed in increaser:

                if speed > upper_border:
                    break

                print("INFO: Speed increases by 10")
                self.current_speed = speed

        else:
            try:
                print("INFO: Speed increases by 10")
                self.current_speed = next(increaser)
            except StopIteration:
                pass

        print(
            f"INFO: The speed of this car has been increased from {start_speed} to {self.current_speed}"
        )

    def brake(self, lower_border=None, step=10):

        start_speed = self.current_speed

        decreaser = DecreaseSpeed(self.current_speed, 0, step)

        if lower_border is not None:

            for speed in decreaser:

                if speed < lower_border:
                    break

                print("INFO: Speed decreases by 10")
                self.current_speed = speed

        else:
            try:
                print("INFO: Speed decreases by 10")
                self.current_speed = next(decreaser)
            except StopIteration:
                pass

        print(
            f"INFO: The speed of this car has been decreased from {start_speed} to {self.current_speed}"
        )

    def parking(self):

        start_speed = self.current_speed

        if self.current_speed != 0:
            self.current_speed = 0

        print(
            f"INFO: The speed of this car has been decreased from {start_speed} to 0"
        )

        if self.state == "on_road":
            print("Parking the car...")
            self.state = "parking"
            Car.cars_on_road -= 1

    @classmethod
    def total_cars(cls):
        return cls.cars_on_road

    @staticmethod
    def show_weather():

        openmeteo = openmeteo_requests.Client()

        url = "https://api.open-meteo.com/v1/forecast"

        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": [
                "temperature_2m",
                "apparent_temperature",
                "rain",
                "wind_speed_10m"
            ],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }

        response = openmeteo.weather_api(url, params=params)[0]
        current = response.Current()

        temp = current.Variables(0).Value()
        apparent = current.Variables(1).Value()
        rain = current.Variables(2).Value()
        wind = current.Variables(3).Value()

        print(f"Current temperature: {round(temp,1)} C")
        print(f"Current apparent_temperature: {round(apparent,1)} C")
        print(f"Current rain: {rain} mm")
        print(f"Current wind_speed: {round(wind,1)} m/s")

In [3]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [4]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [5]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [6]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [7]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 10


In [8]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [9]:
car3.accelerate(80)
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current temperature: 2.3 C
Current apparent_temperature: -1.7 C
Current rain: 0.0 mm
Current wind_speed: 3.6 m/s
Total cars on road: 2


In [10]:
car2.accelerate(10)
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [11]:
Car.show_weather()

Current temperature: 2.3 C
Current apparent_temperature: -1.7 C
Current rain: 0.0 mm
Current wind_speed: 3.6 m/s
